# <font color='#2E86C1' size=6>**Customer table (`clientes.csv`): full preprocessing**</font>

**Workflow:** [01 Data cleaning](01_data_cleaning.ipynb) → **02 Customer table** → [03 Missing values](03_missing_values.ipynb) → [04 Outliers](04_outlier_detection.ipynb) → [05 Preprocessing recipe and models](05_preprocessing_and_models.ipynb) → [06 Feature engineering and selection](06_feature_engineering_selection.ipynb)

`clientes.csv` is **master data**: one row per customer, with geography, company name, address and the year the customer was registered. It has no target and no purchase dates. Every step below is therefore a **rule on the static table**: it reads no purchase row and no label, so it runs identically for training, test.csv and new records. The rules live in `config.yaml` → `customers`, the code in `src/customers.py` → `prepare_customers()`, and the join to the purchases in `src/cleaning.py` → `add_customer_features()`.

The exploration notebook already established:
- `customer_id` is a unique key, and every train customer is in the table;
- `customer_type` is 100% empty;
- two company names appear twice;
- several geographic fields are often blank, with high cardinality.

This notebook turns that into a complete, evidence-based pipeline and adds what the exploration didn't check:
- the labels are codes plus filler text;
- the geography is a strict hierarchy, which allows exact recovery;
- a missing island is structural ("not an island region");
- `customer_since_year` = 2015 means "already a customer when the records start";
- whether each step actually helps the models.

## Table of Contents <a class="anchor" id="toc"></a>
- [0. Setup](#setup)
- [1. What the table contains](#content)
- [2. Validation and the join key](#validation)
- [3. Duplicates and near-duplicates](#duplicates)
- [4. Empty columns and labels → codes](#codes)
- [5. The geographic hierarchy: checks and exact recovery](#hierarchy)
- [6. Island: structural, not missing](#island)
- [7. Cardinality and rare levels](#cardinality)
- [8. `customer_since_year`](#since)
- [9. Customer-level signals](#signals)
- [10. The full pipeline](#pipeline)
- [11. Joining the customer table to the purchases](#join)
- [12. Does it help the models?](#models)
- [13. Summary: from this notebook to the pipeline](#summary)

# <font color='#2E86C1' size=6>**0. Setup**</font> <a class="anchor" id="setup"></a>

[Back to TOC](#toc)

In [1]:
import copy
import os
import sys
from pathlib import Path

# run from the project root, whether Jupyter was started there or in notebooks/
ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml

from src.cleaning import drop_training_rows, prepare_table
from src.customers import (
    check_customer_key, duplicate_report, hierarchy_report, labels_to_codes, prepare_customers,
    profile_completeness, recover_from_hierarchy, structural_fill, rare_level_backoff,
)
from src.data import load_data, standardize_columns, validate_schema
from src.evaluate import cross_validate_pipeline, paired_comparison, select_scoring, temporal_folds
from src.pipeline import build_pipeline, prepare_task_data
from src.tasks import TASKS

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
with open("config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)
data_cfg, cust_cfg = config["data"], config["customers"]
tokens = config["diagnostics"]["placeholder_tokens"]

clientes_raw = load_data(data_cfg["clientes_path"], tokens)
train_raw = load_data(data_cfg["train_path"], tokens)
test_raw = load_data(data_cfg["test_path"], tokens)
train_raw = train_raw[train_raw["Total a Pagar"].notna()]          # the 65 almost-empty rows are not purchases
used_ids = set(train_raw["customer_id"]) | set(test_raw["customer_id"])
clientes = standardize_columns(clientes_raw, {})

# the purchase tables after the stateless rules, with this table joined (notebook 01, section 5.4; section 11)
train_rules, _ = prepare_table(load_data(data_cfg["train_path"], tokens), config, clientes_raw)
train_rules, _ = drop_training_rows(train_rules, config["diagnostics"], data_cfg["id_column"], cust_cfg)
test_rules, _ = prepare_table(test_raw, config, clientes_raw)


# <font color='#2E86C1' size=6>**1. What the table contains**</font> <a class="anchor" id="content"></a>

[Back to TOC](#toc)

In [2]:
profile = pd.DataFrame({
    "missing_% (all 1,143)": clientes.isna().mean().mul(100).round(1),
    "missing_% (629 modelled)": clientes[clientes["customer_id"].isin(used_ids)].isna().mean().mul(100).round(1),
    "distinct values": clientes.nunique(),
    "label pattern (digits -> <n>)": [
        clientes[c].dropna().astype(str).str.replace(r"\d+", "<n>", regex=True).value_counts().index[0]
        if clientes[c].notna().any() else "" for c in clientes.columns],
})
display(profile)
print(f"customers in clientes.csv: {len(clientes)}  |  with purchases in train.csv or test.csv: "
      f"{clientes['customer_id'].isin(used_ids).sum()}  |  only in test.csv: "
      f"{len(set(test_raw['customer_id']) - set(train_raw['customer_id']))}")

,"missing_% (all 1,143)",missing_% (629 modelled),distinct values,label pattern (digits -> <n>)
customer_id,0.0,0.0,1143,<n>
country_or_region,0.0,0.0,11,Country CT<n>: Land of The Final Version
commercial_region,10.0,2.1,37,Region RG<n>: We Will See Later
island,50.6,47.5,10,Island IS<n>: The Pivot Archipelago
municipality,13.9,7.0,183,Municipality MU<n>: Council of Last Minute Edits
city,47.8,38.3,417,City CY<n>: PowerPoint New Hills
postal_area,6.2,6.8,271,Postal area PA<n>: Hidden Sheet Sector
postal_code,14.9,3.2,828,Postal code PC<n>: #REF! Delivery Route
street,52.3,54.4,511,Street ST<n>: Last Save Walk
address,0.5,0.0,1120,Address AD<n>: Cell Without a Home


customers in clientes.csv: 1143  |  with purchases in train.csv or test.csv: 629  |  only in test.csv: 22


- **Three kinds of information:** geography (country → commercial region → island → municipality → city → postal area → postal code → street → address), identity (`company_name`) and history (`customer_since_year`).
- **Every label is "<prefix> <code>: <filler text>"**, e.g. `Region RG17: We Will See Later`. The filler is a joke text; the exploration's "Excel `#REF!` errors" in `postal_code` are part of that filler, not broken cells (section 4).
- **Only 629 of the 1,143 customers have purchases** in train.csv or test.csv. Missing values are much rarer among them: region 2.1% (10% overall), municipality 7.0% (13.9%). The island is the exception at 47.5% (section 6).

# <font color='#2E86C1' size=6>**2. Validation and the join key**</font> <a class="anchor" id="validation"></a>

`validate_schema()`, the same function used for the purchase files, checks the customer table against `config.yaml` → `customers.schema`:
- every column is present;
- ids and years are whole numbers;
- `customer_since_year` lies between 1900 and the **current year**: a registration can't be in the future. The lower limit only rejects absurd years, since an account opened before the records start (2015) is plausible. An out-of-range year is impossible, so it becomes `NaN`, and `years_as_customer` is then imputed inside the pipeline;
- every label follows its pattern.

`check_customer_key()` then stops the pipeline if an id is missing or repeated, because a repeated key would duplicate purchases on the join.

[Back to TOC](#toc)

In [3]:
validated, schema_report = validate_schema(clientes, cust_cfg["schema"])
display(schema_report.groupby("step")["values"].sum().to_frame("values flagged"))
check_customer_key(validated, cust_cfg["key"])
print("customer_id: present and unique for all", len(validated), "customers -> a many-to-one join can't duplicate purchases")

,values flagged
step,
schema: label not allowed -> NaN,0
schema: non-integer value,0
schema: outside valid range -> NaN,0


customer_id: present and unique for all 1143 customers -> a many-to-one join can't duplicate purchases


Every column, type, year and label is valid, and the key is unique: the table can be joined safely.

# <font color='#2E86C1' size=6>**3. Duplicates and near-duplicates**</font> <a class="anchor" id="duplicates"></a>

[Back to TOC](#toc)

In [4]:
display(pd.DataFrame(duplicate_report(clientes, cust_cfg["key"], cust_cfg["near_duplicate_columns"])))
same_company = clientes[clientes["company_name"].duplicated(keep=False)].sort_values("company_name")
same_company = same_company.assign(has_purchases=same_company["customer_id"].isin(used_ids))
display(same_company[["customer_id", "company_name", "customer_since_year", "commercial_region", "municipality", "address", "has_purchases"]])
shared_address = clientes[clientes["address"].notna() & clientes["address"].duplicated(keep=False)]
different_names = shared_address.groupby("address")["company_name"].nunique().gt(1)
print(f"{len(shared_address)} customers share an address with another customer "
      f"({shared_address['address'].nunique()} addresses); different company names at "
      f"{int(different_names.sum())} of those addresses")

,step,column,values,rule
0,customers: exact duplicate customers (ignoring...,,0,
1,customers: customers sharing a value,company_name,4,report only
2,customers: customers sharing a value,address,33,report only


,customer_id,company_name,customer_since_year,commercial_region,municipality,address,has_purchases
390,255774,Excel Is Not A Database Ltd 384,2021,Region RG17: We Will See Later,Municipality MU12: Council of Last Minute Edits,Address AD414: Cell Without a Home,True
391,714679,Excel Is Not A Database Ltd 384,2017,Region RG17: We Will See Later,Municipality MU12: Council of Last Minute Edits,Address AD414: Cell Without a Home,True
998,652673,Excel Is Not A Database Ltd 664,2015,Region RG33: We Will See Later,Municipality MU158: Council of Last Minute Edits,Address AD975: Cell Without a Home,False
999,821383,Excel Is Not A Database Ltd 664,2015,NaN,NaN,Address AD1047: Cell Without a Home,False


33 customers share an address with another customer (16 addresses); different company names at 15 of those addresses


- **No exact duplicates.**
- **Two company names appear twice:**
  - **Ltd 384:** two accounts at the **same address**, registered in 2017 and 2021, and **both have purchases**. That looks like the same company opening a second account. They stay separate: the purchase files identify customers by id, and merging would change the key.
  - **Ltd 664:** probably a duplicate profile, as the exploration suspected, but **neither id has purchases**, so it can't affect the models.
- **33 customers share 16 addresses.** Apart from the Ltd 384 pair, these are different companies at the same address (e.g. a business centre). Section 9 shows they behave like everyone else.

**Decision:** report only (`near_duplicate_columns`); never merge ids.

# <font color='#2E86C1' size=6>**4. Empty columns and labels → codes**</font> <a class="anchor" id="codes"></a>

[Back to TOC](#toc)

In [5]:
empty = [c for c in clientes.columns if clientes[c].isna().all()]
print("100% missing (dropped):", empty)

code_columns = cust_cfg["code_columns"]
codes = {c: clientes[c].astype("string").str.extract(cust_cfg["code_pattern"], expand=False) for c in code_columns}
check = pd.DataFrame({
    "labels": [clientes[c].nunique() for c in code_columns],
    "codes": [codes[c].nunique() for c in code_columns],
    "codes with more than one label": [int((clientes[c].groupby(codes[c]).nunique() > 1).sum()) for c in code_columns],
    "example": [f"{clientes[c].dropna().iloc[0]}  ->  {codes[c].dropna().iloc[0]}" for c in code_columns],
}, index=code_columns)
display(check)

100% missing (dropped): ['customer_type']


,labels,codes,codes with more than one label,example
country_or_region,11,11,0,Country CT11: Land of The Final Version -> CT11
commercial_region,37,37,0,Region RG37: We Will See Later -> RG37
island,10,10,0,Island IS2: The Pivot Archipelago -> IS2
municipality,183,183,0,Municipality MU14: Council of Last Minute Edit...
city,417,417,0,City CY417: Missing Cell Port -> CY417
postal_area,271,271,0,Postal area PA271: Hidden Sheet Sector -> PA271
postal_code,828,828,0,Postal code PC828: #REF! Delivery Route -> P...
street,511,511,0,Street ST170: PowerPoint New Walk -> ST170
address,1120,1120,0,Address AD1120: Cell Without a Home -> AD1120


- **`customer_type` carries no information** (100% missing), so it is dropped. The rule drops any column that is entirely empty, so a future file is handled the same way.
- **Each code always comes with the same label**, and the number of codes equals the number of labels. So keeping only the code (`RG17`, `PC459`, …) **loses nothing**, and it makes the categories short and readable.
- **No postal code is broken:** the `#REF! Delivery Route` text appears in every postal code label, so it is filler, not an error.

`labels_to_codes()` raises an error if a code ever comes with two labels, so codes can never silently merge different values.

# <font color='#2E86C1' size=6>**5. The geographic hierarchy: checks and exact recovery**</font> <a class="anchor" id="hierarchy"></a>

If every value of a lower level belongs to exactly one value of a higher level, a missing higher level can be **recovered exactly** from the lower one. This is the same idea as the identity rules for purchases (notebook 01, section 5.1).

[Back to TOC](#toc)

In [6]:
coded, _ = labels_to_codes(clientes.drop(columns=empty), code_columns, cust_cfg["code_pattern"])
pairs = []
levels = ["country_or_region", "commercial_region", "island", "municipality", "city", "postal_area", "postal_code"]
for child in levels:
    for parent in levels:
        if child == parent:
            continue
        both = coded[[child, parent]].dropna()
        n_parents = both.groupby(child)[parent].nunique()
        if len(both) and (n_parents == 1).mean() >= 0.97:
            pairs.append({"child": child, "parent": parent, "rows with both": len(both),
                          "child codes with one parent (%)": round(100 * (n_parents == 1).mean(), 1),
                          "conflicting child codes": int((n_parents > 1).sum())})
display(pd.DataFrame(pairs))

,child,parent,rows with both,child codes with one parent (%),conflicting child codes
0,commercial_region,country_or_region,1029,100.0,0
1,island,country_or_region,565,100.0,0
2,island,commercial_region,565,100.0,0
3,municipality,country_or_region,984,100.0,0
4,municipality,commercial_region,984,100.0,0
5,municipality,island,565,100.0,0
6,city,country_or_region,597,100.0,0
7,city,commercial_region,574,100.0,0
8,city,island,354,100.0,0
9,city,municipality,546,100.0,0


In [7]:
recovered, recovery_rows = recover_from_hierarchy(coded, cust_cfg["hierarchy_recovery"])
display(pd.DataFrame(recovery_rows)[["column", "rule", "values"]].rename(columns={"rule": "recovered", "values": "customers"}))
for column in ["commercial_region", "municipality"]:
    print(f"{column}: missing {coded[column].isna().sum()} -> {recovered[column].isna().sum()} "
          f"(modelled customers: {coded.loc[coded['customer_id'].isin(used_ids), column].isna().sum()} -> "
          f"{recovered.loc[recovered['customer_id'].isin(used_ids), column].isna().sum()})")

,column,recovered,customers
0,commercial_region,from postal_area,69
1,commercial_region,from postal_code,2
2,commercial_region,from municipality,0
3,commercial_region,from city,0
4,municipality,from postal_code,5
5,municipality,from city,0


commercial_region: missing 114 -> 43 (modelled customers: 13 -> 9)
municipality: missing 159 -> 154 (modelled customers: 44 -> 44)


- **The geography is a strict hierarchy:** every island, municipality, postal area and postal code belongs to exactly one region, and every region to one country. The only conflicts are 6 postal areas that span two municipalities and a few cities across postal areas. Those pairs are reported (`hierarchy_checks`) and **never used for recovery**.
- **Recovered exactly:**
  - **commercial region: 71 of 114 gaps**, mostly from the postal area (13 → 9 among the modelled customers);
  - **municipality: 5**, from the postal code (none among the modelled customers).

  The rest have no lower level to recover from, and stay missing for the fitted imputer.
- Like the identity rules, this reads only the customer's own record and the static table, so it is a stateless rule.

# <font color='#2E86C1' size=6>**6. Island: structural, not missing**</font> <a class="anchor" id="island"></a>

[Back to TOC](#toc)

In [8]:
by_region = recovered.groupby(recovered["commercial_region"].fillna("(unknown)")).agg(
    customers=("customer_id", "size"), with_island=("island", lambda s: s.notna().mean()))
print("regions where every customer has an island:", list(by_region.index[by_region["with_island"] == 1]))
print("regions where no customer has an island:", int((by_region["with_island"] == 0).sum()),
      "| regions with a mix:", int(((by_region["with_island"] > 0) & (by_region["with_island"] < 1)).sum()))
filled, island_rows = structural_fill(recovered, cust_cfg["structural_fill"])
display(pd.DataFrame(island_rows)[["column", "step", "values", "rule"]])
print("island after the fill:", filled["island"].value_counts(dropna=False).head(12).to_dict())

regions where every customer has an island: ['RG17', 'RG29']
regions where no customer has an island: 36 | regions with a mix: 0


,column,step,values,rule
0,island,customers: structural fill -> 'none',535,commercial_region has no island


island after the fill: {'none': 535, 'IS7': 133, 'IS8': 90, 'IS4': 88, 'IS6': 81, nan: 43, 'IS1': 43, 'IS9': 43, 'IS5': 37, 'IS2': 26, 'IS3': 21, 'IS10': 3}


**Islands only exist in two commercial regions (RG17 and RG29).** In each of the other regions, *no* customer has an island, and there is no region with a mix.

So a blank island means **"this customer is not in an island region"**: the value is undefined, not unknown. This corrects the earlier reading in notebook 03, section 3, which treated the 34% gap as MAR "related to the target". The relation to the target was simply the island vs mainland regions.

- **Island gaps:** become **`"none"`** where the region is known (`structural_fill`, 535 customers).
- **Region also unknown:** the island stays missing (43 customers).
- **An island region with a missing island:** the value would be a real gap and stay missing (0 cases).

The model now sees a proper "mainland" category instead of an imputed one.

# <font color='#2E86C1' size=6>**7. Cardinality and rare levels**</font> <a class="anchor" id="cardinality"></a>

[Back to TOC](#toc)

In [9]:
modelled = filled[filled["customer_id"].isin(used_ids)]
rows = []
for column in ["country_or_region", "commercial_region", "island", "municipality", "city", "postal_area", "postal_code", "street", "address"]:
    counts = modelled[column].value_counts()
    rows.append({"column": column, "levels (modelled customers)": len(counts),
                 "levels with < 5 customers": int((counts < 5).sum()),
                 "customers in those levels": int(counts[counts < 5].sum()),
                 "missing %": round(100 * modelled[column].isna().mean(), 1)})
display(pd.DataFrame(rows))
backed_off, backoff_rows = rare_level_backoff(filled, {
    "municipality": {"min_customers": 10, "fallback_column": "commercial_region"},
    "country_or_region": {"min_customers": 10, "fallback": "other"}})
print("with a back-off at 10 customers: municipality", filled["municipality"].nunique(), "->",
      backed_off["municipality"].nunique(), "levels; country", filled["country_or_region"].nunique(), "->",
      backed_off["country_or_region"].nunique())

,column,levels (modelled customers),levels with < 5 customers,customers in those levels,missing %
0,country_or_region,7,5,8,0.0
1,commercial_region,32,15,27,1.4
2,island,11,1,2,1.4
3,municipality,154,129,224,7.0
4,city,271,259,302,38.3
5,postal_area,195,172,266,6.8
6,postal_code,535,535,609,3.2
7,street,271,271,287,54.4
8,address,618,618,629,0.0


with a back-off at 10 customers: municipality 183 -> 39 levels; country 11 -> 3


**Which geographic levels are kept:**
- **Kept:** country, commercial region, island and municipality.
- **Dropped:** city, postal area, postal code, street and address. Among the 629 modelled customers they have 195–618 levels, mostly with 1–4 customers each. They behave like a customer id and add no geography that region and municipality don't already carry.
- **Street** is also more than half empty.
- **`company_name`** is unique per customer (an id), so it is dropped too.

**Municipality is still fine-grained:** 129 of its 154 levels have fewer than 5 modelled customers. Grouping small municipalities into their region (`rare_level_backoff`) is implemented. Section 12 tests whether it helps.

# <font color='#2E86C1' size=6>**8. `customer_since_year`**</font> <a class="anchor" id="since"></a>

[Back to TOC](#toc)

In [10]:
purchases = pd.concat([train_raw[["customer_id", "purchase_date"]], test_raw[["customer_id", "purchase_date"]]])
first_purchase = purchases.assign(date=pd.to_datetime(purchases["purchase_date"])).groupby("customer_id")["date"].min()
since = clientes.set_index("customer_id")["customer_since_year"]
print("customers registered per year:", since.value_counts().sort_index().to_dict())
joined = pd.DataFrame({"since": since, "first_purchase_year": first_purchase.dt.year}).dropna()
print("customers whose first purchase is BEFORE their registration year:", int((joined["first_purchase_year"] < joined["since"]).sum()))
print("2016+ cohorts: first purchase in the registration year:",
      f"{(joined.loc[joined['since'] >= 2016, 'first_purchase_year'] == joined.loc[joined['since'] >= 2016, 'since']).mean():.0%}")
print("2015 'cohort': first purchase year:", joined.loc[joined["since"] == 2015, "first_purchase_year"].value_counts().sort_index().to_dict())

customers registered per year: {2015: 664, 2016: 53, 2017: 82, 2018: 61, 2019: 47, 2020: 50, 2021: 52, 2022: 43, 2023: 29, 2024: 31, 2025: 18, 2026: 13}
customers whose first purchase is BEFORE their registration year: 0
2016+ cohorts: first purchase in the registration year: 67%
2015 'cohort': first purchase year: {2015.0: 161, 2016.0: 26, 2017.0: 15, 2018.0: 13, 2019.0: 5, 2020.0: 7, 2021.0: 5, 2022.0: 4, 2023.0: 1, 2024.0: 1, 2025.0: 1}


- **The year is consistent:** no customer buys before their registration year. The join still checks it for every purchase (`years_as_customer >= 0`, `customers.consistency_rules`), so new data is covered too.
- **2015 is not a normal cohort.** 664 of 1,143 customers are dated 2015, against 13–82 in every later year, and 2015 is the first year of the records (the first purchase is 2015-07-11). Many of those "2015" customers only buy for the first time years later, while 67% of later cohorts make their first purchase in their registration year.

So `customer_since_year = 2015` means **"already a customer when the records start"**, with an unknown true start. Two consequences:
- **`years_as_customer`** (purchase year − registration year, computed at each purchase) **under-states** tenure for these customers.
- **A flag, `customer_since_before_records`**, lets a model tell them apart.

# <font color='#2E86C1' size=6>**9. Customer-level signals**</font> <a class="anchor" id="signals"></a>

Before testing them in the models: do the candidate customer features relate to the targets at all? (Purchase level, all training rows.)

[Back to TOC](#toc)

In [11]:
customers_clean, _ = prepare_customers(clientes_raw, cust_cfg)
extra = clientes[["customer_id", "address"]].assign(
    shares_address=lambda d: d["address"].notna() & d["address"].duplicated(keep=False))
d = (train_raw.merge(customers_clean, on="customer_id", how="left")
     .merge(extra[["customer_id", "shares_address"]], on="customer_id", how="left"))
d["island_region"] = d["island"].ne("none") & d["island"].notna()
for feature in ["customer_since_before_records", "island_region", "shares_address", "profile_completeness"]:
    table = d.groupby(feature).agg(purchases=("ID", "size"), customers=("customer_id", "nunique"),
                                   repurchase_rate=("repeat_purchase_90d", "mean"),
                                   mean_future_spend=("spend_next_90d", "mean")).round(3)
    print(f"\n{feature}")
    display(table)


customer_since_before_records


,purchases,customers,repurchase_rate,mean_future_spend
customer_since_before_records,,,,
0,2937,369,0.517,1532.604
1,3532,238,0.596,2200.647



island_region


,purchases,customers,repurchase_rate,mean_future_spend
island_region,,,,
False,2189,284,0.641,2027.029
True,4280,323,0.518,1857.471



shares_address


,purchases,customers,repurchase_rate,mean_future_spend
shares_address,,,,
False,6289,583,0.560,1921.432
True,180,24,0.567,1925.095



profile_completeness


,purchases,customers,repurchase_rate,mean_future_spend
profile_completeness,,,,
1,28,7,0.393,2567.867
2,101,15,0.416,2275.973
3,155,18,0.561,916.479
4,966,107,0.551,1391.285
5,3748,313,0.579,2169.673
6,1471,147,0.530,1665.510


- **Already a customer when the records start:** these customers repurchase more often (0.60 vs 0.52) and spend more when they return (≈ 2,200 € vs ≈ 1,530 €). They are the long-standing accounts. **Candidate feature.**
- **Island vs mainland region:** 0.52 vs 0.64 repurchase. Region and island already carry this.
- **Shared address:** no difference (0.567 vs 0.560). **Reported, not a feature.**
- **Profile completeness:** customers with only 1–2 address fields repurchase less (≈ 0.40, few purchases) than complete profiles (≈ 0.55). A data-quality signal, likely newer or less engaged accounts. **Candidate feature.**

# <font color='#2E86C1' size=6>**10. The full pipeline**</font> <a class="anchor" id="pipeline"></a>

`prepare_customers()` runs every step in order and reports every value it changed. `prepare_table()` joins the result to every purchase table.

[Back to TOC](#toc)

In [12]:
customers_clean, customer_report = prepare_customers(clientes_raw, cust_cfg)
display(customer_report[customer_report["values"] > 0])
display(customers_clean.head())
print("missing values in the joined columns (all 1,143 customers):", customers_clean.isna().sum().to_dict())

,step,column,values,rule
1,customers: customers sharing a value,company_name,4,report only
2,customers: customers sharing a value,address,33,report only
3,customers: 100% missing -> dropped,customer_type,1143,
4,customers: label -> code,country_or_region,1143,([A-Z]{2}\d+):
5,customers: label -> code,commercial_region,1029,([A-Z]{2}\d+):
6,customers: label -> code,island,565,([A-Z]{2}\d+):
7,customers: label -> code,municipality,984,([A-Z]{2}\d+):
8,customers: label -> code,city,597,([A-Z]{2}\d+):
9,customers: label -> code,postal_area,1072,([A-Z]{2}\d+):
10,customers: label -> code,postal_code,973,([A-Z]{2}\d+):


,customer_id,country_or_region,commercial_region,island,municipality,profile_completeness,customer_since_before_records,customer_since_year
0,842570.0,CT11,RG37,none,NaN,3,0,2018.0
1,770941.0,CT9,NaN,NaN,NaN,1,0,2021.0
2,827053.0,CT10,RG17,IS2,MU14,5,0,2016.0
3,494648.0,CT10,RG17,IS2,MU14,4,1,2015.0
4,429291.0,CT10,RG17,IS2,MU14,4,1,2015.0


missing values in the joined columns (all 1,143 customers): {'customer_id': 0, 'country_or_region': 0, 'commercial_region': 43, 'island': 43, 'municipality': 154, 'profile_completeness': 0, 'customer_since_before_records': 0, 'customer_since_year': 0}


**The output:** one row per customer, with:
- 4 geography codes (country, region, island with `"none"`, municipality);
- 2 new features (`profile_completeness`, `customer_since_before_records`);
- the registration year.

At the join, the registration year becomes `years_as_customer` at the moment of each purchase, and the year itself is dropped. The remaining gaps are real ones, imputed inside the model pipeline (regression: explicit `"Missing"` category, notebook 03, section 4):
- region: 43 customers, 9 of them modelled;
- the matching island values;
- municipality: 154 customers.

# <font color='#2E86C1' size=6>**11. Joining the customer table to the purchases**</font> <a class="anchor" id="join"></a>

[Back to TOC](#toc)

In [13]:
customers = standardize_columns(clientes_raw, {})
customer_profile = pd.DataFrame({"missing_%": customers.isna().mean().mul(100).round(1), "distinct_values": customers.nunique()})
display(customer_profile)
print("training purchases whose customer is not in clientes.csv:", int((~train_rules["customer_id"].isin(customers["customer_id"])).sum()),
      "| test:", int((~test_rules["customer_id"].isin(customers["customer_id"])).sum()))
print("purchases made before the customer's customer_since_year:", int((train_rules["years_as_customer"] < 0).sum()),
      "| test:", int((test_rules["years_as_customer"] < 0).sum()))

,missing_%,distinct_values
customer_id,0.0,1143
country_or_region,0.0,11
commercial_region,10.0,37
island,50.6,10
municipality,13.9,183
city,47.8,417
postal_area,6.2,271
postal_code,14.9,828
street,52.3,511
address,0.5,1120


training purchases whose customer is not in clientes.csv: 0 | test: 0
purchases made before the customer's customer_since_year: 0 | test: 0


Every customer is found, and no purchase is dated before the customer's `customer_since_year`, so the join adds no future information. Columns (`config.yaml` → `customers`):
- **kept:** `country_or_region`, `commercial_region`, `island`, `municipality` (geography at a usable level of detail), plus `customer_since_year`, turned into `years_as_customer` at the moment of the purchase;
- **dropped:**
  - `customer_type` is 100% missing.
  - `company_name` and `address` are unique per customer, so they would act as a customer id.
  - `street` and `city` are about half missing and have hundreds of levels.
  - `postal_code` and `postal_area` have hundreds of levels, and the kept geography already covers them.

The join is a stateless, row-preserving step (`add_customer_features()`), so it also runs on test.csv.

**The full preprocessing of the customer table** is in sections 1–10 above (`src/customers.py`):
- validation and key checks;
- labels → codes;
- exact recovery of missing regions through the geographic hierarchy;
- `island` as a structural `"none"` outside the island regions;
- the "customer before the records" flag and profile completeness, which the regression now also uses.

Notebook 06, section 1.2 tests the customer table as introduced here in the models: the four geography columns and `years_as_customer`.

# <font color='#2E86C1' size=6>**12. Does it help the models?**</font> <a class="anchor" id="models"></a>

The variants below are scored on the same 5 temporal folds, with each task's chosen recipe and model, and compared fold by fold (notebook 01, section 7):
- **A, previous pipeline:** raw labels, no recovery, a blank island imputed like any gap.
- **B, + codes + exact recovery + island "none".**
- **C, B + the two new features.** This is the current configuration.
- **D, C + municipality back-off** (< 10 customers → its region).

[Back to TOC](#toc)

In [14]:
def customers_variant(**changes):
    variant = copy.deepcopy(config)
    variant["customers"].update(changes)
    return variant

previous = customers_variant(code_columns=[], hierarchy_recovery=[], structural_fill={}, rare_level_backoff={})
backoff = customers_variant(rare_level_backoff={"municipality": {"min_customers": 10, "fallback_column": "commercial_region"},
                                                "country_or_region": {"min_customers": 10, "fallback": "other"}})
new_features = ["profile_completeness", "customer_since_before_records"]

def without(recipe, features):
    return {**recipe, "numeric_features": [f for f in recipe["numeric_features"] if f not in features]}

task = "regression"
metric = config[task]["selection_metric"]
scoring = select_scoring(TASKS[task].evaluate.SCORING, config[task]["metrics"])
scores = {}
for name, cfg, drop in [("A previous pipeline", previous, new_features), ("B codes + recovery + island none", config, new_features),
                        ("C B + new features (current)", config, []), ("D C + municipality back-off", backoff, [])]:
    data = prepare_task_data(cfg, task)
    folds = temporal_folds(data["extras_dev"][data_cfg["date_column"]], config["cv"]["n_splits"], config["cv"]["gap_days"])
    recipe = without(data["preprocessing_config"], drop)
    pipeline = build_pipeline(recipe, TASKS[task].model.build_model(config[task]["model"]))
    scores[name] = cross_validate_pipeline(pipeline, data["X_dev"], data["y_dev"], folds, scoring)[f"validation_{metric}"].to_numpy()

display(pd.DataFrame({"mean RMSE": {k: v.mean() for k, v in scores.items()}, "std": {k: v.std(ddof=1) for k, v in scores.items()}}).round(1))
display(pd.DataFrame([{"A": a, "B": b, **paired_comparison(scores[a], scores[b], higher_is_better=False)}
                      for a, b in [("B codes + recovery + island none", "A previous pipeline"),
                                   ("C B + new features (current)", "B codes + recovery + island none"),
                                   ("C B + new features (current)", "A previous pipeline"),
                                   ("D C + municipality back-off", "C B + new features (current)")]]).round(2))

,mean RMSE,std
A previous pipeline,1942.8,163.5
B codes + recovery + island none,1943.2,164.5
C B + new features (current),1942.5,165.2
D C + municipality back-off,1943.6,163.6


,A,B,mean_advantage,se,folds_a_better,n_folds,decisive
0,B codes + recovery + island none,A previous pipeline,-0.40,0.90,3,5,False
1,C B + new features (current),B codes + recovery + island none,0.68,2.32,3,5,False
2,C B + new features (current),A previous pipeline,0.27,1.89,3,5,False
3,D C + municipality back-off,C B + new features (current),-1.14,1.91,2,5,False


In [15]:
task = "classification"
metric = config[task]["selection_metric"]
scoring = select_scoring(TASKS[task].evaluate.SCORING, config[task]["metrics"])
data = prepare_task_data(config, task)
folds = temporal_folds(data["extras_dev"][data_cfg["date_column"]], config["cv"]["n_splits"], config["cv"]["gap_days"])
recipe = data["preprocessing_config"]
customer_features = {**recipe,
                     "numeric_features": recipe["numeric_features"] + ["years_as_customer"] + new_features,
                     "categorical_features": recipe["categorical_features"] + ["country_or_region", "commercial_region", "island", "municipality"]}
class_scores = {name: cross_validate_pipeline(build_pipeline(r, TASKS[task].model.build_model(config[task]["model"])),
                                              data["X_dev"], data["y_dev"], folds, scoring)[f"validation_{metric}"].to_numpy()
                for name, r in {"without customer features (current)": recipe, "with the cleaned customer features": customer_features}.items()}
print({k: round(v.mean(), 4) for k, v in class_scores.items()})
print(paired_comparison(class_scores["with the cleaned customer features"], class_scores["without customer features (current)"]))

{'without customer features (current)': np.float64(0.7078), 'with the cleaned customer features': np.float64(0.7002)}
{'mean_advantage': np.float64(-0.007642934334750473), 'se': np.float64(0.0036143673407317365), 'folds_a_better': 1, 'n_folds': 5, 'decisive': True}


**Verdict.** Compared with the fold-to-fold spread (≈ ±165 RMSE), every difference is small, and none is decisive. With the imputation of notebook 03 in place (customer-history fill, row limits, `spend_365d` estimated from its row), they are now within about 1 RMSE: B vs A −0.4 (better on 3 of 5 folds), C vs B +0.7 (3 of 5), D vs C −1.1 (2 of 5). Earlier, with the simpler imputation, B and C gained 1.5 and 2.3 RMSE. With the rule of notebook 01, section 7 (change for a decisive gain, or for a clear reason that most folds confirm):

- **Codes, exact recovery and island `"none"`: kept.**
  - Codes carry exactly the same information, so they score the same, and they are clean and readable.
  - Recovery and the structural fill make the data **correct**: a value recovered exactly instead of imputed, and an undefined island no longer treated as unknown. On the models they are now neutral (better on 3 of 5 folds, 0.4 RMSE behind on average), so they are kept for that correctness, not for a score.
- **`profile_completeness` and `customer_since_before_records`: added to the regression** (`regression.preprocessing.extra_numeric_features`). They have a clear reason (section 9) and are slightly ahead (0.7 RMSE, 3 of 5 folds).
- **Municipality back-off: implemented but off** (`rare_level_backoff: {}`). It isn't better: the target encoder already shrinks small municipalities towards the overall mean. It stays one config line away for encoders that need it (one-hot).
- **Classification: still without customer features.** Even cleaned, they make logistic regression decisively worse (−0.008 accuracy, SE 0.004, better on 1 of 5 folds), as in notebook 06, section 1.2.

# <font color='#2E86C1' size=6>**13. Summary: from this notebook to the pipeline**</font> <a class="anchor" id="summary"></a>

| Step | Finding | Rule (`config.yaml` → `customers`) | Code (`src/customers.py`) |
|---|---|---|---|
| 1. Load | blank cells are the only placeholder | `diagnostics.placeholder_tokens` | `src/data.load_data` |
| 2. Validate | all columns, types, years and labels valid | `schema` | `validate_schema` |
| 3. Key | `customer_id` unique → safe many-to-one join | `key` | `check_customer_key` |
| 4. Duplicates | no exact duplicates; 2 company names twice (one pair with purchases: kept as separate accounts); 33 customers share addresses | `near_duplicate_columns` (report) | `duplicate_report` |
| 5. Empty columns | `customer_type` 100% missing → dropped | automatic | `prepare_customers` |
| 6. Labels → codes | one label per code; `#REF!` is filler | `code_pattern`, `code_columns` | `labels_to_codes` |
| 7. Profile completeness | fewer address fields → lower repurchase | `profile_columns` | `profile_completeness` |
| 8. Hierarchy | strict hierarchy; conflicts reported, never used | `hierarchy_checks` | `hierarchy_report` |
| 9. Exact recovery | region 71, municipality 5 | `hierarchy_recovery` | `recover_from_hierarchy` |
| 10. Structural fill | island undefined outside RG17 / RG29 → `"none"` (535) | `structural_fill` | `structural_fill` |
| 11. Rare levels | back-off not better → off | `rare_level_backoff` | `rare_level_backoff` |
| 12. Flags | `customer_since_year` 2015 = before the records start | `flags` | `prepare_customers` |
| 13. Columns kept | country, region, island, municipality, completeness, flag, year; city / postal / street / address / company name dropped (id-like) | `columns` | `prepare_customers` |
| 14. Join | left join, every purchase kept; `years_as_customer` at the purchase; no purchase before registration | `derived_features`, `consistency_rules` | `src/cleaning.add_customer_features` |
| 15. Model use | regression: geography + years + 2 new features; classification: none | `regression.preprocessing`, `classification.preprocessing` | — |

[Back to TOC](#toc)